In [3]:
from langchain_core.documents import Document

In [4]:
doc=Document(
    page_content="this is the main text that will be used for the rag",
    metadata={
        "source":"dummy.txt",
        "pages":10,
        "author":"Sohan",
        
    }
)

In [5]:
doc

Document(metadata={'source': 'dummy.txt', 'pages': 10, 'author': 'Sohan'}, page_content='this is the main text that will be used for the rag')

In [7]:
import os
os.makedirs("../data/text_files",exist_ok=True)

In [9]:
sample_texts={
    "../data/text_files/python_intro.txt":"""Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
Created by Guido van Rossum and first released in 1991, Python has become one of the most popular
programming languages in the world.

Key Features:
- Easy to learn and use
- Extensive standard library
- Cross-platform compatibility
- Strong community support

Python is widely used in web development, data science, artificial intelligence, and automation.""",
    
    "../data/text_files/machine_learning.txt": """Machine Learning Basics

Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing computer programs
that can access data and use it to learn for themselves.

Types of Machine Learning:
1. Supervised Learning: Learning with labeled data
2. Unsupervised Learning: Finding patterns in unlabeled data
3. Reinforcement Learning: Learning through rewards and penalties

Applications include image recognition, speech processing, and recommendation systems
    
    
    """

}
for filepath,content in sample_texts.items():
    with open(filepath,"w",encoding="utf-8") as f:
        f.write(content)
print("created and written to files")

created and written to files


In [10]:
from langchain_community.document_loaders import TextLoader

In [11]:
loader=TextLoader("../data/text_files/python_intro.txt",encoding="utf-8")

In [14]:
document=loader.load()
document

[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nCreated by Guido van Rossum and first released in 1991, Python has become one of the most popular\nprogramming languages in the world.\n\nKey Features:\n- Easy to learn and use\n- Extensive standard library\n- Cross-platform compatibility\n- Strong community support\n\nPython is widely used in web development, data science, artificial intelligence, and automation.')]

In [15]:
from langchain_community.document_loaders import DirectoryLoader

In [16]:
dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="**/*.txt",
    show_progress=True,
    loader_cls=TextLoader
)

In [18]:
documents=dir_loader.load()
documents[0]

100%|██████████| 2/2 [00:00<00:00, 1003.06it/s]


Document(metadata={'source': '..\\data\\text_files\\machine_learning.txt'}, page_content='Machine Learning Basics\n\nMachine learning is a subset of artificial intelligence that enables systems to learn and improve\nfrom experience without being explicitly programmed. It focuses on developing computer programs\nthat can access data and use it to learn for themselves.\n\nTypes of Machine Learning:\n1. Supervised Learning: Learning with labeled data\n2. Unsupervised Learning: Finding patterns in unlabeled data\n3. Reinforcement Learning: Learning through rewards and penalties\n\nApplications include image recognition, speech processing, and recommendation systems\n    \n    \n    ')

In [19]:
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader

In [20]:
dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="**/*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=True
)

In [22]:
pdf_docs=dir_loader.load()
pdf_docs

100%|██████████| 1/1 [00:00<00:00,  1.43it/s]


[Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2025-12-30T13:45:32+00:00', 'source': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'file_path': '..\\data\\pdf\\LOS_Question_Bank.pdf', 'total_pages': 1, 'format': 'PDF 1.4', 'title': '(anonymous)', 'author': '(anonymous)', 'subject': '(unspecified)', 'keywords': '', 'moddate': '2025-12-30T13:45:32+00:00', 'trapped': '', 'modDate': "D:20251230134532+00'00'", 'creationDate': "D:20251230134532+00'00'", 'page': 0}, page_content='Linux Operating Systems (LOS)\nQuestion Bank – 5 & 10 Marks\n5 MARK QUESTIONS\n1\nDefine a signal in Linux. List any four commonly used signals.\n2\nExplain the purpose of signal() system call.\n3\nWhat is sigaction()? How is it better than signal()?\n4\nDifferentiate between kill() and raise().\n5\nExplain thread and its advantages.\n6\nExplain pthread_create() and pthread_join().\n7\nWhat is mutex? Why is it used?\n8\nDefine semaphore and list its

In [23]:
from pathlib import Path

In [24]:
def process_all_pdf_files(pdf_directory):
    all_docs=[]
    pdf_dir=Path(pdf_directory)
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    print(f"found {len(pdf_files)} files to process")

    for pdf_file in pdf_files:
        try:
            print(f"loading : {pdf_file.name}")
            loader=PyMuPDFLoader(str(pdf_file))
            documents=loader.load()

            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'
            all_docs.append(doc)
        except Exception as e:
            print(f"error : {e}")
    print(f"total number of documents : {len(all_docs)}")
    return all_docs

all_pdf_documents=process_all_pdf_files("../data")

found 1 files to process
loading : LOS_Question_Bank.pdf
total number of documents : 1


Embedding store and vectorDB

In [27]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from typing import List,Tuple,Any,Dict
from sklearn.metrics.pairwise import cosine_similarity

c:\Users\LENONO\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
100%|██████████| 1/1 [21:48<00:00, 1308.62s/it]


In [28]:
class EmbeddingManager:
    def __init__(self,model_name:str="all-MiniLM-L6-V2"):
        self.model_name=model_name
        self.model=None
        self._load_model()

    def _load_model(self):
        try:
            self.model=SentenceTransformer(self.model_name)
            print(f"model loaded successfully")
        except Exception as e:
            print(f"failed to load model : {self.model_name} : {e}")
            raise
    
    def generate_embeddings(self,texts:List[str])->np.ndarray:
        if not self.model:
            raise ValueError("Model not loaded")
        
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with dimensions : {embeddings.shape}")
        return embeddings
    
embedding_manager=EmbeddingManager()

c:\Users\LENONO\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\LENONO\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-V2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3226.41it/s]
Bert

model loaded successfully


Here we store in the VectorDB

In [33]:
class VectorStore:
    def __init__(self,collection_name:str="pdf_documents",persistent_dir:str="../data/vector_store"):
        self.collection_name=collection_name
        self.persistent_dir=persistent_dir
        self.collection=None
        self.client=None
        self._initialize_store()
    
    def _initialize_store(self):
        try:
            os.makedirs(self.persistent_dir,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persistent_dir)

            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"desc":"pdf embeddings for RAG"}
            )
            print(f"vector store initialized :{self.collection_name}")
            print(f"existing documents in the collection : {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store : {e}")
            raise
    
    def add_documents(self,documents:List[Any],embeddings:np.ndarray):
        if len(documents)!=len(embeddings):
            raise ValueError("Number of documents must be equal to the number of embeddings")
        
        print(f"adding {len(documents)} documents to the vector store")

        ids=[]
        metadatas=[]
        documents_text=[]
        embeddings_list=[]

        for i,(doc,embedding) in enumerate(zip(documents,embeddings)):
            doc_id=f"doc_{uuid.uuid64().hex[:8]}_{i}"
            ids.append(doc_id)

            metadata=dict(doc.metadata)
            metadata['doc_index']=i
            metadata['content_length']=len(doc.page_content)
            metadatas.append(metadata)

            documents_text.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"add {len(documents)} documents to the vector store")
        
        except Exception as e:
            print(f"error storing the documents into the vector store : {e}")
            raise

vectorstore=VectorStore()

vector store initialized :pdf_documents
existing documents in the collection : 0


In [34]:
vectorstore